# Predicciones de Kaggle con MiniLM
Usamos las premisas e hipótesis originales en los 15 idiomas. MiniLM genera un vector por frase y combinamos cuatro bloques: premisa, hipótesis, diferencia absoluta y producto. MLP y XGBoost se ajustan con las etiquetas del train oficial. La predicción es la clase con mayor probabilidad media.

La validación local agrupada por premisa dio 62,14 % de accuracy y 61,84 % de macro-F1 de media en tres semillas. Este cuaderno genera el CSV con todo el train. La celda final comprueba el formato y el orden oficial.

In [ ]:
import hashlib
import json
import sys
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

KAGGLE_INPUT = Path("/kaggle/input")
ON_KAGGLE = KAGGLE_INPUT.is_dir()
PROJECT = Path.cwd()
DATA_DIR = PROJECT / "data"
MODEL_DIR = PROJECT / ".local" / "minilm-checkpoint"
OUTPUT_DIR = PROJECT / ".local" / "minilm_submission"

if ON_KAGGLE:
    OUTPUT_DIR = Path("/kaggle/working")
    sources = list(KAGGLE_INPUT.rglob("minilm_submission.py"))
    if len(sources) == 1:
        PROJECT = sources[0].parent
    else:
        archives = list(KAGGLE_INPUT.rglob("minilm-source.zip"))
        if len(archives) != 1:
            raise FileNotFoundError("Adjunta el paquete de código minilm-source.zip.")
        source_workspace = TemporaryDirectory()
        PROJECT = Path(source_workspace.name)
        with ZipFile(archives[0]) as archive:
            archive.extractall(PROJECT)

    datasets = [p.parent for p in KAGGLE_INPUT.rglob("sample_submission.csv")
                if (p.parent / "train.csv").is_file() and (p.parent / "test.csv").is_file()]
    checkpoints = [p.parent for p in KAGGLE_INPUT.rglob("pytorch_model.bin")
                   if (p.parent / "modules.json").is_file()]
    if len(datasets) != 1 or len(checkpoints) != 1:
        raise FileNotFoundError("Adjunta los datos oficiales de la competición y el dataset público de MiniLM.")
    DATA_DIR, MODEL_DIR = datasets[0], checkpoints[0]

sys.path.insert(0, str(PROJECT))
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from threadpoolctl import threadpool_limits

from classifiers import build_classifier
from minilm_submission import FIXED_TOKENIZER, MODEL_SHA256, TEST_ROWS, prepare_checkpoint, validate_submission, verify_official_data
from replay_dense_rf import _full_features
from training import write_submission

## Datos y representación

Verificamos los tres archivos oficiales y ordenamos las filas de test según la plantilla. Codificamos cada texto distinto una vez. El encoder está congelado y el test solo se usa para predecir.

In [ ]:
verify_official_data(DATA_DIR)
options = {"dtype": {"id": str}, "keep_default_na": False}
train = pd.read_csv(DATA_DIR / "train.csv", **options)
test = pd.read_csv(DATA_DIR / "test.csv", **options)
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", **options)
if (len(train) != 12120 or len(test) != TEST_ROWS or len(sample) != TEST_ROWS
        or not all(frame.id.is_unique for frame in (train, test, sample))
        or set(test.id) != set(sample.id) or set(train.label) != {0, 1, 2}):
    raise ValueError("Las filas, los IDs o las etiquetas oficiales no coinciden.")
test = test.set_index("id").loc[sample.id].reset_index()

protocol = json.loads((PROJECT / "results_dense_generalization/protocol.json").read_text(encoding="utf-8"))
rows = pd.concat([train, test], ignore_index=True)
texts = pd.concat([rows.premise, rows.hypothesis], ignore_index=True)
codes, unique = pd.factorize(texts)

# Usamos CPU como en la ejecución de Kaggle con el tokenizador corregido.
torch.set_num_threads(8)
with TemporaryDirectory() as model_workspace:
    patched_model = Path(model_workspace)
    prepare_checkpoint(MODEL_DIR, patched_model)
    encoder = SentenceTransformer(str(patched_model), device="cpu", local_files_only=True, trust_remote_code=False)
    tokenizer = encoder.tokenizer
    effective_tokenizer = {
        "class": type(tokenizer).__name__,
        "backend": type(tokenizer.backend_tokenizer.model).__name__,
        "max_seq_length": encoder.max_seq_length,
        "pooling": encoder[1].get_config_dict(),
    }
    expected = {
        "class": "TokenizersBackend", "backend": "Unigram", "max_seq_length": 128,
        "pooling": {"embedding_dimension": 384, "pooling_mode": "mean", "include_prompt": True},
    }
    if effective_tokenizer != expected:
        raise ValueError(f"MiniLM cargó otra configuración: {effective_tokenizer}")
    vectors = encoder.encode(unique.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    del encoder

if vectors.shape != (len(unique), 384) or not np.isfinite(vectors).all():
    raise ValueError("Dimensiones o valores no válidos en los embeddings.")
features = _full_features(*np.split(vectors[codes], 2))
x_train, x_test = features[:len(train)], features[len(train):]

## Ajuste y exportación

Entrenamos con semilla 42. Usamos los parámetros del protocolo de validación y el mismo dispositivo de la ejecución corregida de Kaggle. Escribimos el CSV después de validar las probabilidades y volvemos a comprobarlo desde disco antes de moverlo a la salida.

In [ ]:
SEED = 42
probabilities = []
configs = {}
families = {"mlp_256_alpha0.0003": "mlp", "xgb_d3_500": "xgboost"}

with threadpool_limits(limits=8):
    for tag in protocol["ensemble_2"]:
        family = families[tag]
        params = dict(protocol["configs"][tag])
        if family == "xgboost":
            params["device"] = "cpu"
        configs[tag] = params
        print(f"Ajustando {tag}, semilla {SEED}", flush=True)
        model = build_classifier(family, random_state=SEED, **params)
        model.fit(x_train, train.label.to_numpy())
        np.testing.assert_array_equal(model.classes_, [0, 1, 2])
        proba = model.predict_proba(x_test)
        if proba.shape != (TEST_ROWS, 3) or not np.isfinite(proba).all() or (proba < 0).any():
            raise ValueError(f"Probabilidades inválidas en {tag}.")
        np.testing.assert_allclose(proba.sum(axis=1), 1, atol=1e-5)
        probabilities.append(proba)

predictions = np.mean(probabilities, axis=0).argmax(axis=1)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(dir=OUTPUT_DIR) as staging:
    candidate = Path(staging) / "submission.csv"
    write_submission(test.id, predictions, candidate)
    validate_submission(candidate, DATA_DIR / "sample_submission.csv")
    output = candidate.replace(OUTPUT_DIR / "submission.csv")

record = {
    "seed": SEED,
    "train_rows": len(train),
    "test_rows": len(test),
    "model": protocol["encoder"],
    "features": protocol["features"],
    "tokenizer": effective_tokenizer,
    "tokenizer_config_sha256": FIXED_TOKENIZER,
    "weights_sha256": MODEL_SHA256["pytorch_model.bin"],
    "configs": configs,
    "versions": {pkg: version(pkg) for pkg in ("torch", "transformers", "tokenizers", "sentence-transformers",
                                              "numpy", "pandas", "scipy", "scikit-learn", "xgboost")},
    "submission_sha256_lf": hashlib.sha256(output.read_bytes().replace(b"\r\n", b"\n")).hexdigest(),
}
(OUTPUT_DIR / "minilm_run.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
print(f"CSV válido: {len(predictions)} filas con IDs en el orden oficial. Salida: {output}")